# CCP Default Waterfall: Parametric Simulation

This notebook implements a compact CCP simulation for instructional and exploratory use. It models daily P&L for clearing members on a single underlying, initial margin via parametric VaR, and a simplified default waterfall (member IM → CCP capital → mutualized DF → capped assessments → residual shortfall).

## Setup

In [ ]:
# Runtime: Python 3.x. Colab-ready.
# Dependencies: numpy, pandas, matplotlib, scipy
import math
import random
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Reproducibility
np.random.seed(42)
random.seed(42)

## Configuration

In [ ]:
# Market (single underlying, GBM)
SIM_DAYS = 252
S0 = 5000.0
MU_ANNUAL = 0.06
SIGMA_ANNUAL = 0.20
DT = 1/252

# Clearing members
N_MEMBERS = 20
MIN_EQUITY = 50_000_000.0
MAX_EQUITY = 200_000_000.0
MIN_NOTIONAL = -2_000_000_000.0
MAX_NOTIONAL =  2_000_000_000.0

# Margin and default fund
IM_CONF = 0.99
IM_HORIZON_DAYS = 5
DF_RATIO_TO_IM = 0.10
CCP_SKIN_IN_THE_GAME = 100_000_000.0

# Default test buffer (set >0 to require headroom above IM+DF)
DEFAULT_TRIGGER_BUFFER = 0.0

# Assessments (cap as % of survivors' IM)
ASSESSMENT_CAP_RATIO = 0.20

# Plotting
PLOT_FIGSIZE = (8, 4)

## Model

In [ ]:
@dataclass
class Market:
    s0: float
    mu_annual: float
    sigma_annual: float
    dt: float

    def simulate_path(self, days: int) -> np.ndarray:
        z = np.random.normal(size=days)
        inc = (self.mu_annual - 0.5*self.sigma_annual**2)*self.dt + self.sigma_annual*math.sqrt(self.dt)*z
        s = np.empty(days+1)
        s[0] = self.s0
        s[1:] = self.s0 * np.exp(np.cumsum(inc))
        return s

def gaussian_var(sigma_annual: float, horizon_days: int, alpha: float) -> float:
    sigma_h = sigma_annual * math.sqrt(horizon_days/252)
    q = norm.ppf(1-alpha)   # negative; VaR uses absolute value
    return -q * sigma_h

@dataclass
class ClearingMember:
    name: str
    equity: float
    notional: float
    im: float = 0.0
    df_contrib: float = 0.0
    defaulted: bool = False

    def pnl(self, daily_return: float) -> float:
        return self.notional * daily_return

class CCP:
    def __init__(self, members: List[ClearingMember], ccp_capital: float):
        self.members = members
        self.ccp_capital = ccp_capital
        self.history = []  # snapshots per day

    def compute_im(self, sigma_annual: float, horizon_days: int, alpha: float) -> None:
        unit_var = gaussian_var(sigma_annual, horizon_days, alpha)
        for m in self.members:
            m.im = abs(m.notional) * unit_var
            m.df_contrib = DF_RATIO_TO_IM * m.im

    def total_df(self) -> float:
        return sum(m.df_contrib for m in self.members if not m.defaulted)

    def settle_vm_and_check_defaults(self, daily_return: float):
        defaults = []
        total_uncovered = 0.0

        # VM settlement
        for m in self.members:
            if m.defaulted: 
                continue
            m.equity += m.pnl(daily_return)

        # Default test
        for m in self.members:
            if m.defaulted: 
                continue
            required = m.im + m.df_contrib + DEFAULT_TRIGGER_BUFFER
            if m.equity < required:
                m.defaulted = True
                defaults.append(m)

        # Residual loss (beyond IM + equity)
        for m in defaults:
            residual = max(-(m.equity) - m.im, 0.0)
            total_uncovered += residual

        return defaults, total_uncovered

    def apply_waterfall(self, uncovered: float):
        usage = {"ccp_capital": 0.0, "mutualized_df": 0.0, "assessment": 0.0, "shortfall": 0.0}
        remaining = uncovered

        # CCP capital
        take = min(self.ccp_capital, remaining)
        self.ccp_capital -= take
        usage["ccp_capital"] = take
        remaining -= take
        if remaining <= 0:
            return usage

        # Mutualized DF (pro-rata)
        survivors = [m for m in self.members if not m.defaulted]
        total_df = sum(m.df_contrib for m in survivors)
        take_df = min(total_df, remaining)
        usage["mutualized_df"] = take_df
        remaining -= take_df
        if total_df > 0 and take_df > 0:
            for m in survivors:
                w = m.df_contrib / total_df
                m.df_contrib -= w * take_df
        if remaining <= 0:
            return usage

        # Assessments (capped)
        cap_total = ASSESSMENT_CAP_RATIO * sum(m.im for m in survivors)
        take_assess = min(cap_total, remaining)
        usage["assessment"] = take_assess
        remaining -= take_assess
        total_im = sum(m.im for m in survivors)
        if total_im > 0 and take_assess > 0:
            for m in survivors:
                w = m.im / total_im
                m.equity -= w * take_assess

        usage["shortfall"] = max(remaining, 0.0)
        return usage

    def snapshot(self, day: int, defaults, usage) -> None:
        self.history.append({
            "day": day,
            "ccp_capital": self.ccp_capital,
            "total_df": self.total_df(),
            "defaults": [m.name for m in defaults],
            "waterfall_usage": usage,
            "member_equities": {m.name: m.equity for m in self.members},
            "member_df": {m.name: m.df_contrib for m in self.members},
        })

## Initialization

In [ ]:
def init_members(n: int) -> List[ClearingMember]:
    members = []
    for i in range(n):
        equity = float(np.random.uniform(MIN_EQUITY, MAX_EQUITY))
        notional = float(np.random.uniform(MIN_NOTIONAL, MAX_NOTIONAL))
        members.append(ClearingMember(name=f"CM_{i+1}", equity=equity, notional=notional))
    return members

market = Market(S0, MU_ANNUAL, SIGMA_ANNUAL, DT)
members = init_members(N_MEMBERS)
ccp = CCP(members, CCP_SKIN_IN_THE_GAME)
ccp.compute_im(SIGMA_ANNUAL, IM_HORIZON_DAYS, IM_CONF)

s_path = market.simulate_path(SIM_DAYS)
daily_returns = np.diff(s_path) / s_path[:-1]

print(f"members={len(members)}  initial_total_df={ccp.total_df():,.0f}")

## Run

In [ ]:
def run(ccp: CCP, daily_returns: np.ndarray) -> None:
    d = 0
    for r in daily_returns:
        d += 1
        defaults, uncovered = ccp.settle_vm_and_check_defaults(r)
        usage = ccp.apply_waterfall(uncovered) if defaults else {"ccp_capital":0.0,"mutualized_df":0.0,"assessment":0.0,"shortfall":0.0}
        ccp.snapshot(d, defaults, usage)

run(ccp, daily_returns)
print("done")

## Results

In [ ]:
hist = pd.DataFrame(ccp.history)
hist['num_defaults'] = hist['defaults'].apply(len)

plt.figure(figsize=PLOT_FIGSIZE)
plt.plot(s_path)
plt.title("Underlying Path (GBM)")
plt.xlabel("Day")
plt.ylabel("Level")
plt.show()

plt.figure(figsize=PLOT_FIGSIZE)
plt.plot(hist['day'], hist['num_defaults'])
plt.title("Defaults per Day")
plt.xlabel("Day")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=PLOT_FIGSIZE)
plt.plot(hist['day'], hist['ccp_capital'], label="CCP capital")
plt.plot(hist['day'], hist['total_df'], label="Total DF")
plt.title("CCP Capital and Default Fund")
plt.xlabel("Day")
plt.ylabel("Amount")
plt.legend()
plt.show()

def sum_usage(series, key):
    return series.apply(lambda d: d.get(key, 0.0) if isinstance(d, dict) else 0.0)

usage_df = pd.DataFrame({
    "day": hist["day"],
    "ccp_capital": sum_usage(hist["waterfall_usage"], "ccp_capital"),
    "mutualized_df": sum_usage(hist["waterfall_usage"], "mutualized_df"),
    "assessment": sum_usage(hist["waterfall_usage"], "assessment"),
    "shortfall": sum_usage(hist["waterfall_usage"], "shortfall"),
})

plt.figure(figsize=PLOT_FIGSIZE)
plt.plot(usage_df['day'], usage_df['ccp_capital'], label="skin-in-the-game")
plt.plot(usage_df['day'], usage_df['mutualized_df'], label="mutualized DF")
plt.plot(usage_df['day'], usage_df['assessment'], label="assessment")
plt.plot(usage_df['day'], usage_df['shortfall'], label="shortfall")
plt.title("Waterfall Utilization (Daily)")
plt.xlabel("Day")
plt.ylabel("Amount")
plt.legend()
plt.show()

summary = pd.DataFrame({
    "final_ccp_capital": [hist['ccp_capital'].iloc[-1]],
    "final_total_df": [hist['total_df'].iloc[-1]],
    "total_defaults": [hist['num_defaults'].sum()],
    "days_with_any_default": [(hist['num_defaults'] > 0).sum()],
})
summary

## Stress (optional)

In [ ]:
def stress_shock(ccp: CCP, shock_return: float):
    defaults, uncovered = ccp.settle_vm_and_check_defaults(shock_return)
    usage = ccp.apply_waterfall(uncovered) if defaults else {"ccp_capital":0.0,"mutualized_df":0.0,"assessment":0.0,"shortfall":0.0}
    last_day = ccp.history[-1]['day'] if ccp.history else 0
    ccp.snapshot(last_day + 1, defaults, usage)
    return defaults, usage

# Example:
# stress_shock(ccp, shock_return=-0.10)